# Crypto Volatility EDA\n**Rizaldy Utomo**  \n`rutomo@andrew.cmu.edu`\n\n- This notebook profiles engineered one-second crypto volatility features.\n- The goal is to inspect label balance, feature behavior, and threshold selection before modeling.\n- Figures are exported to the project-level `img/` folder for reuse in reports.\n

## Setup and Reproducibility\n

In [ ]:
from pathlib import Path\nimport json\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport seaborn as sns\nfrom IPython.display import Markdown, display\n\nBASE_DIR = Path('..')\nIMG_DIR = BASE_DIR / 'img'\nIMG_DIR.mkdir(parents=True, exist_ok=True)\nFEATURE_PATH = BASE_DIR / 'data/processed/features.parquet'\n\nsns.set_theme(style='whitegrid')\npd.set_option('display.max_columns', 30)\n

## Helper Functions\n

In [ ]:
def save_and_show(fig, filename):\n    output_path = IMG_DIR / filename\n    fig.savefig(output_path, dpi=150, bbox_inches='tight')\n    plt.show()\n    plt.close(fig)\n\ndef interpretation_block(title, lines):\n    body = '\\n'.join([f'- {line}' for line in lines])\n    display(Markdown(f'### {title}\\n{body}'))\n

## Load Feature Data\n

In [ ]:
features = pd.read_parquet(FEATURE_PATH)\nfeatures['window_end_ts'] = pd.to_datetime(features['window_end_ts'], utc=True)\nfeatures = features.sort_values(['product_id', 'window_end_ts']).reset_index(drop=True)\ndisplay(features.head())\ndisplay(features[['product_id', 'label']].groupby('product_id').agg(['count', 'mean']))\n

## Light EDA\nThis pass keeps the analysis short and focused on the behavior of the volatility label and the main engineered feature groups.\n

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))\nsns.histplot(features['sigma_future_60s'].dropna(), bins=40, color='steelblue', ax=ax)\nax.set_title('Future 60s Volatility Distribution')\nax.set_xlabel('sigma_future_60s')\nax.set_ylabel('Count')\nsave_and_show(fig, 'eda_sigma_future_distribution.png')\n

The future-volatility label source should be right-skewed. The exact tail behavior matters because the threshold is defined from the upper quantiles.\n

In [ ]:
tau_candidates = features['sigma_future_60s'].quantile([0.80, 0.85, 0.90, 0.95]).rename('tau').reset_index()\ntau_candidates.columns = ['quantile', 'tau']\ntau_candidates['positive_rate'] = tau_candidates['tau'].apply(lambda t: float((features['sigma_future_60s'] >= t).mean()))\ndisplay(tau_candidates)\n\nfig, ax = plt.subplots(figsize=(9, 4.5))\nax.plot(tau_candidates['quantile'], tau_candidates['positive_rate'], marker='o', color='darkorange')\nax.set_title('Positive Class Rate by Tau Quantile')\nax.set_xlabel('Quantile')\nax.set_ylabel('Positive Rate')\nsave_and_show(fig, 'eda_tau_positive_rate.png')\n

This percentile sweep is the key threshold-selection step. The 90th percentile is the default unless it makes the event rate too sparse for a stable classifier.\n

In [ ]:
sample = features[['spread_bps', 'realized_vol_60s', 'ewma_abs_return', 'label']].dropna().sample(min(1000, len(features)), random_state=42)\nfig, axes = plt.subplots(1, 2, figsize=(13, 5))\nsns.scatterplot(data=sample, x='spread_bps', y='realized_vol_60s', hue='label', palette='Set1', alpha=0.65, ax=axes[0])\naxes[0].set_title('Spread vs Realized Volatility')\nsns.boxplot(data=sample, x='label', y='ewma_abs_return', ax=axes[1])\naxes[1].set_title('EWMA Absolute Return by Label')\nfig.tight_layout()\nsave_and_show(fig, 'eda_feature_relationships.png')\n

These feature relationships are a quick check that the positive label is at least directionally aligned with wider spreads and larger recent return magnitudes.\n

## Final Summary\n- Confirm that the chosen `tau` produces a usable positive-class rate.\n- Reuse the exported figures in `img/` for the model evaluation report.\n- If label balance is unstable, revisit the tau percentile before training.\n